In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


PROJECT_ROOT = r"C:\Users\aakif\Documents\DataCompetition"
TRAIN_PATH = PROJECT_ROOT + r"\data\train.csv"

TARGET = "Will_Buy_EV"
ID_COL = "id"

train = pd.read_csv(TRAIN_PATH)

X = train.drop(columns=[TARGET, ID_COL]).copy()
y = train[TARGET].map({"No": 0, "Yes": 1}).astype(int)

print("Rows:", len(train))
print("")
print("Columns:")
print(X.columns.tolist())
print("")


formula_columns = [
    "Annual_Income_USD",
    "Environmental_Concern_Level",
    "Subsidy_Available",
    "Range_Anxiety_Level"
]

print("Formula columns:")
for col in formula_columns:
    print("")
    print("###", col)
    print(X[col].value_counts(dropna=False).sort_index())


def normalize_subsidy(series):
    """
    Convert common Yes/No representations into 0/1.
    """
    s = series.astype("string").str.strip().str.lower()

    return s.map({
        "yes": 1.0,
        "no": 0.0,
        "true": 1.0,
        "false": 0.0,
        "1": 1.0,
        "0": 0.0
    }).fillna(0.0)


def encode_range_anxiety(series):
    """
    Map the reported low/medium/high range-anxiety structure.
    """
    s = series.astype("string").str.strip().str.lower()

    return s.map({
        "low": 0.0,
        "medium": 1.0,
        "high": 3.0
    }).fillna(1.0)


def evaluate_score(score, label):
    auc = roc_auc_score(y, score)

    print("")
    print("------------------------------------------------------------")
    print(label)
    print("------------------------------------------------------------")
    print(f"ROC-AUC: {auc:.6f}")

    return auc


income = (
    X["Annual_Income_USD"]
    .fillna(X["Annual_Income_USD"].median())
    / 100000.0
)

environment = (
    X["Environmental_Concern_Level"]
    .fillna(X["Environmental_Concern_Level"].median())
)

subsidy = normalize_subsidy(
    X["Subsidy_Available"]
)

range_anxiety = encode_range_anxiety(
    X["Range_Anxiety_Level"]
)


formula_score = (
    1.2 * income
    + 0.6 * environment
    + 2.0 * subsidy
    - 1.0 * range_anxiety
)

score_32a = evaluate_score(
    formula_score,
    "32A_Reported_Generator_Formula"
)

range_anxiety_linear = (
    X["Range_Anxiety_Level"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "low": 0.0,
        "medium": 1.0,
        "high": 2.0
    })
    .fillna(1.0)
)

formula_score_linear = (
    1.2 * income
    + 0.6 * environment
    + 2.0 * subsidy
    - range_anxiety_linear
)

score_32b = evaluate_score(
    formula_score_linear,
    "32B_Linear_Range_Anxiety_Formula"
)


formula_frame = pd.DataFrame({
    "income": income,
    "environment": environment,
    "subsidy": subsidy,
    "range_anxiety": range_anxiety
})

logistic = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic.fit(
    formula_frame,
    y
)

logistic_probability = logistic.predict_proba(
    formula_frame
)[:, 1]

score_32c = evaluate_score(
    logistic_probability,
    "32C_Fitted_Four_Feature_Logistic"
)

print("")
print("Learned logistic coefficients:")
for name, coef in zip(
    formula_frame.columns,
    logistic.coef_[0]
):
    print(f"{name}: {coef:.6f}")

print("")
print("Intercept:", float(logistic.intercept_[0]))

formula_raw = X[formula_columns].copy()

for col in formula_columns:
    formula_raw[f"{col}__missing"] = (
        formula_raw[col].isna().astype(int)
    )

categorical_formula = [
    "Subsidy_Available",
    "Range_Anxiety_Level"
]

numeric_formula = [
    "Annual_Income_USD",
    "Environmental_Concern_Level"
] + [
    f"{col}__missing"
    for col in formula_columns
]

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            )
        ]),
        numeric_formula
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_formula
    )
])

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(
    formula_raw,
    y
)

logistic_full_probability = model.predict_proba(
    formula_raw
)[:, 1]

score_32d = evaluate_score(
    logistic_full_probability,
    "32D_Four_Feature_Logistic_With_Missingness"
)


# ============================================================
# RESULTS
# ============================================================

results = pd.DataFrame([
    {
        "Experiment": "32A_Reported_Generator_Formula",
        "ROC_AUC": score_32a
    },
    {
        "Experiment": "32B_Linear_Range_Anxiety_Formula",
        "ROC_AUC": score_32b
    },
    {
        "Experiment": "32C_Fitted_Four_Feature_Logistic",
        "ROC_AUC": score_32c
    },
    {
        "Experiment": "32D_Four_Feature_Logistic_With_Missingness",
        "ROC_AUC": score_32d
    }
])

results = results.sort_values(
    "ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("")
print("")
print("============================================================")
print("EXPERIMENT 32 RESULTS")
print("============================================================")
print(results.to_string(index=False))

best_score = float(
    results.iloc[0]["ROC_AUC"]
)

print("")
print("23B / 31A benchmark: 0.945243")
print(f"Best Experiment 32: {best_score:.6f}")
print(
    f"Difference vs 23B: "
    f"{best_score - 0.945243:+.6f}"
)

print("")
print("No submission generated.")
print("No prediction CSV generated.")
print("Experiment 32 complete.")

Rows: 668665

Columns:
['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

Formula columns:

### Annual_Income_USD
Annual_Income_USD
30000.0     61605
31003.0         2
38174.0         1
38209.0         1
38250.0         1
            ...  
184807.0        1
186488.0        1
186936.0        1
187822.0        2
188549.0        1
Name: count, Length: 13214, dtype: int64

### Environmental_Concern_Level
Environmental_Concern_Level
1.0    147476
2.0    135133
3.0    127351
4.0    130469
5.0    128236
Name: count, dtype: int64

### Subsidy_Available
Subsidy_Available
No     248756
Yes    419909
Name: count, dtype: int64

### Range_Anxiety_Level
Range_Anxiety_Level
High        2194
Low       603972
Medium     62499
Name: count, dtype: int64

-------------------------